In [1]:
from ultralytics import YOLO
from ultralytics.data.dataset import YOLODataset
import ultralytics.data.build as build
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [2]:
class YOLOWeightedDataset(YOLODataset):
    def __init__(self, *args, mode="train", **kwargs):
        """
        Initialize the WeightedDataset.

        Args:
            class_weights (list or numpy array): A list or array of weights corresponding to each class.
        """

        super(YOLOWeightedDataset, self).__init__(*args, **kwargs)

        self.train_mode = "train" in self.prefix

        # You can also specify weights manually instead
        self.count_instances()
        class_weights = np.sum(self.counts) / self.counts
        self.agg_func = np.mean

        self.class_weights = np.array(class_weights)
        self.weights = self.calculate_weights()
        self.probabilities = self.calculate_probabilities()

    def count_instances(self):
        """
        Count the number of instances per class

        Returns:
            dict: A dict containing the counts for each class.
        """
        self.counts = [0 for i in range(len(self.data["names"]))]
        for label in self.labels:
            cls = label['cls'].reshape(-1).astype(int)
            for id in cls:
                self.counts[id] += 1

        self.counts = np.array(self.counts)
        self.counts = np.where(self.counts == 0, 1, self.counts)

    def calculate_weights(self):
        """
        Calculate the aggregated weight for each label based on class weights.

        Returns:
            list: A list of aggregated weights corresponding to each label.
        """
        weights = []
        for label in self.labels:
            cls = label['cls'].reshape(-1).astype(int)

            # Give a default weight to background class
            if cls.size == 0:
              weights.append(1)
              continue

            # Take mean of weights
            # You can change this weight aggregation function to aggregate weights differently
            # weight = np.mean(self.class_weights[cls])
            # weight = np.max(self.class_weights[cls])
            weight = self.agg_func(self.class_weights[cls])
            weights.append(weight)
        return weights

    def calculate_probabilities(self):
        """
        Calculate and store the sampling probabilities based on the weights.

        Returns:
            list: A list of sampling probabilities corresponding to each label.
        """
        total_weight = sum(self.weights)
        probabilities = [w / total_weight for w in self.weights]
        return probabilities

    def __getitem__(self, index):
        """
        Return transformed label information based on the sampled index.
        """
        # Don't use for validation
        if not self.train_mode:
            return self.transforms(self.get_image_and_label(index))
        else:
            index = np.random.choice(len(self.labels), p=self.probabilities)
            return self.transforms(self.get_image_and_label(index))

In [3]:
build.YOLODataset = YOLOWeightedDataset

In [4]:
from ultralytics import YOLO

# 1. YOLOv8m 사전 학습 모델 로드
model = YOLO("yolov8m.pt")

# 2. 모델 학습 설정
model.train(
    data="/home/kth/robot_dev/mediapipe/data/drivision_final/data.yaml",  # 학습/검증 이미지 및 클래스 정보가 들어있는 YAML 경로
    #resume=False,
    epochs=100,                      # 학습을 50번 반복
    imgsz=640,                      # 입력 이미지 크기 (640x640)
    batch=4,                        # 배치 사이즈 (한 번에 학습할 이미지 수, GPU 용량에 따라 조정)
    # :렌치: 학습률 관련 설정
    lr0=0.001,                      # 초기 learning rate (너무 낮으면 학습 거의 안 됨, focal에선 0.001 추천)
    lrf=0.01,                       # 최종 learning rate (cosine scheduler 끝 값)
    warmup_epochs=3,               # 초기 3 epoch 동안 learning rate를 점진적으로 증가 (수렴 안정화)
    weight_decay=0.0005,           # 과적합 방지를 위한 가중치 감소 계수
    optimizer="SGD",               # 최적화 알고리즘 (SGD: 일반적으로 안정적이며 focal과 잘 어울림)
    # :포장: 데이터 증강 설정 (작은 객체 대응 및 일반화 성능 향상)
    mosaic=1.0,                    # 여러 이미지를 섞어서 구성하는 모자이크 증강 (작은 객체에 매우 유용)
    hsv_h=0.015,                   # 색조 변화 범위
    hsv_s=0.7,                     # 채도 변화 범위
    hsv_v=0.4,                     # 명도 변화 범위
    degrees=0.2,                   # 회전 범위 (±0.2도)
    translate=0.1,                 # 이미지 이동 비율 (10%)
    scale=0.5,                     # 이미지 확대/축소 비율
    shear=0.0,                     # 기울이기 (사용 안 함)
    perspective=0.0,               # 원근 왜곡 (사용 안 함)
    flipud=0.0,                    # 상하 반전 확률 (도로 객체에선 무의미하므로 0)
    fliplr=0.5,                    # 좌우 반전 확률 (차량, 표지판 등에는 유효)
    #cls_weights=[0.23, 5.17, 3.27, 0.89, 5.36, 2.95, 10.0, 3.38, 3.57, 4.53, 3.48],
    augment=True,
    name="change_lossfunction_toASL_model"
)

New https://pypi.org/project/ultralytics/8.3.159 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.146 🚀 Python-3.12.3 torch-2.7.0+cu126 CUDA:0 (NVIDIA GeForce RTX 2060, 5740MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/kth/robot_dev/mediapipe/data/drivision_final/data.yaml, degrees=0.2, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=change

train: Scanning /home/kth/robot_dev/mediapipe/data/drivision_final/train/labels... 12858 images, 441 backgrounds, 0 corrupt: 100%|██████████| 12858/12858 [00:03<00:00, 3365.92it/s]

train: /home/kth/robot_dev/mediapipe/data/drivision_final/train/images/crop001607_jpg.rf.3f2dd63f0891e434b07791ef1fe84d17.jpg: 1 duplicate labels removed
train: /home/kth/robot_dev/mediapipe/data/drivision_final/train/images/crop001607_jpg.rf.5d917e31de4ccdbf5a0469d1383c2e1d.jpg: 1 duplicate labels removed
train: /home/kth/robot_dev/mediapipe/data/drivision_final/train/images/crop001607_jpg.rf.956d92303c814f9bb9f68566a38a3be9.jpg: 1 duplicate labels removed
train: /home/kth/robot_dev/mediapipe/data/drivision_final/train/images/crop001616_jpg.rf.623e9aa320e52dd43e0f40d23582b981.jpg: 3 duplicate labels removed
train: /home/kth/robot_dev/mediapipe/data/drivision_final/train/images/crop001616_jpg.rf.86c35422f8d2c6239b7302c5d0f74535.jpg: 3 duplicate labels removed
train: /home/kth/robot_dev/mediapipe/data/drivision_final/train/images/crop001616_jpg.rf.a125e042f503d158c5e5528577433594.jpg: 3 duplicate labels removed
train: /home/kth/robot_dev/mediapipe/data/drivision_final/train/images/perso

train: New cache created: /home/kth/robot_dev/mediapipe/data/drivision_final/train/labels.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1355.1±732.8 MB/s, size: 106.3 KB)


val: Scanning /home/kth/robot_dev/mediapipe/data/drivision_final/valid/labels... 476 images, 16 backgrounds, 0 corrupt: 100%|██████████| 476/476 [00:00<00:00, 1409.23it/s]

...

                   all        476       2932      0.923      0.901      0.944      0.691



100 epochs completed in 14.144 hours.
Optimizer stripped from runs/detect/change_lossfunction_toASL_model6/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/change_lossfunction_toASL_model6/weights/best.pt, 52.0MB

Validating runs/detect/change_lossfunction_toASL_model6/weights/best.pt...
Ultralytics 8.3.146 🚀 Python-3.12.3 torch-2.7.0+cu126 CUDA:0 (NVIDIA GeForce RTX 2060, 5740MiB)
Model summary (fused): 92 layers, 25,846,129 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 60/60 [00:11<00:00,  5.21it/s]


                   all        476       2932      0.916      0.906      0.943      0.681
                   car        309       1549      0.884      0.789      0.888      0.583
      child_protection         74         78      0.914          1      0.982      0.717
          construction         65        157      0.939      0.875      0.948      0.649
                person        199        509       0.88      0.778       0.85      0.548
        speed_limit_30         82         89      0.934      0.944       0.96      0.675
        speed_limit_50        113        119      0.897      0.975      0.968      0.683
             stop_sign         37         37      0.919          1      0.994      0.819
                veh_go         61        130      0.974      0.874      0.939      0.648
            veh_goLeft         42         76      0.888      0.934      0.967      0.738
              veh_stop         44        103      0.917      0.858      0.928      0.674
           veh_warnin

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x716d850a8ce0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.